### Load data files into BQ tables

#### Helpful links:
*   [BQ Client](https://cloud.google.com/python/docs/reference/bigquery/latest/google.cloud.bigquery.client.Client)
*   [LoadJobConfig](https://cloud.google.com/python/docs/reference/bigquery/latest/google.cloud.bigquery.job.LoadJobConfig)


#### Create BQ dataset for storing the raw data

In [2]:
from google.cloud import bigquery

project_id = "segfault-434120"
dataset = "skincare_products_raw"
region = "us-central1"

bq_client = bigquery.Client()

dataset_id = bigquery.Dataset(f"{project_id}.{dataset}")
dataset_id.location = region
resp = bq_client.create_dataset(dataset_id, exists_ok=True)
print("Created dataset {}.{}".format(bq_client.project, resp.dataset_id))

Created dataset segfault-434120.skincare_products_raw


#### Common functions

In [3]:
from google.cloud import bigquery

project_id = "segfault-434120"
bucket = "skincare-products"
parent_folder = "raw"
region = "us-central1"
dataset = "skincare_products_raw"

bq_client = bigquery.Client()

def create_load_table_from_csv(folder, file_name, table, schema, delimiter=",", quote_character="\"", skip_leading_rows=1):

  uri = f"gs://{bucket}/{parent_folder}/{folder}/{file_name}"
  table_id = f"{project_id}.{dataset}.{table}"

  table = bigquery.Table(table_id, schema=schema)
  table = bq_client.create_table(table, exists_ok=True)
  print("Created table {}".format(table.table_id))

  # remove the data_source and load_time fields before loading the data,
  # neither one is present in the csv
  del schema[-1]
  del schema[-1]
  print(schema)

  job_config = bigquery.LoadJobConfig(
        skip_leading_rows=skip_leading_rows,
        schema=schema,
        source_format=bigquery.SourceFormat.CSV,
        write_disposition="WRITE_TRUNCATE",
        field_delimiter=delimiter,
        quote_character=quote_character,
        allow_jagged_rows=True,
        allow_quoted_newlines=True,
        ignore_unknown_values=True
      )

  load_job = bq_client.load_table_from_uri(uri, table_id, job_config=job_config)
  load_job.result()

  destination_table = bq_client.get_table(table_id)
  print("Loaded {} rows.".format(destination_table.num_rows))



def create_load_table_from_json(folder, file_name, table, schema):

  table_id = f"{project_id}.{dataset}.{table}"

  table = bigquery.Table(table_id, schema=schema)
  table = bq_client.create_table(table, exists_ok=True)
  print("Created table {}".format(table.table_id))

  # remove the data_source and load_time fields before loading the data,
  # neither one is present in the json
  del schema[-1]
  del schema[-1]

  #print(schema)

  job_config = bigquery.LoadJobConfig(schema=schema,
      source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
      write_disposition = "WRITE_TRUNCATE"
  )

  uri = f"gs://{bucket}/{parent_folder}/{folder}/{file_name}"

  load_job = bq_client.load_table_from_uri(
      uri,
      table_id,
      location=region,
      job_config=job_config,
  )

  load_job.result()

  destination_table = bq_client.get_table(table_id)
  print("Loaded {} rows.".format(destination_table.num_rows))


#### Create and load `cosmetic_ingredient`

In [ ]:
folder = "cosmetic_ingredients"
file_name = "*.csv"
table = "cosmetic_ingredient"
delimiter = ","

schema = [
  bigquery.SchemaField("cosing_ref_no", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("ingredient_unique_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("ingredient_common_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("european_pharmacopoeia_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("cas_no", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("ec_no", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("chemical_description", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("restriction", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("function", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("update_date", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("_data_source", "STRING", mode="REQUIRED", default_value_expression="'cosmetic_ingredient'"),
  bigquery.SchemaField("_load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
]

create_load_table_from_csv(folder, file_name, table, schema, delimiter, skip_leading_rows=9) # Skipping metadata rows

Created table cosmetic_ingredient
[SchemaField('cosing_ref_no', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('ingredient_unique_name', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('ingredient_common_name', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('european_pharmacopoeia_name', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('cas_no', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('ec_no', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('chemical_description', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('restriction', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('function', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('update_date', 'STRING', 'NULLABLE', None, None, (), None)]
Loaded 28714 rows.


#### Create and load `sephora_products_reviews`

In [ ]:
folder = "sephora_products_reviews"
file_name = "*.csv"
table = "sephora_product_review"
delimiter = ","



schema = [
  bigquery.SchemaField("row_number", "INTEGER", mode="NULLABLE"), # Unlabled Row
  bigquery.SchemaField("author_id", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("rating", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("is_recommended", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("helpfulness", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("total_feedback_count", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("total_neg_feedback_count", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("total_pos_feedback_count", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("submission_time", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("review_text", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("review_title", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("skin_tone", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("eye_color", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("skin_type", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("hair_color", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("product_id", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("product_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("brand_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("price_usd", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("_data_source", "STRING", mode="REQUIRED", default_value_expression="'sephora_product_review'"),
  bigquery.SchemaField("_load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
]

create_load_table_from_csv(folder, file_name, table, schema, delimiter)

Created table sephora_product_review
[SchemaField('row_number', 'INTEGER', 'NULLABLE', None, None, (), None), SchemaField('author_id', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('rating', 'INTEGER', 'NULLABLE', None, None, (), None), SchemaField('is_recommended', 'FLOAT', 'NULLABLE', None, None, (), None), SchemaField('helpfulness', 'FLOAT', 'NULLABLE', None, None, (), None), SchemaField('total_feedback_count', 'INTEGER', 'NULLABLE', None, None, (), None), SchemaField('total_neg_feedback_count', 'INTEGER', 'NULLABLE', None, None, (), None), SchemaField('total_pos_feedback_count', 'INTEGER', 'NULLABLE', None, None, (), None), SchemaField('submission_time', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('review_text', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('review_title', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('skin_tone', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('eye_color', 'STRING', 'NULLABLE', None, None, (),

#### Create and load `dermatologic_diseases`

Remove leading and trailing whitespace from files

In [ ]:
folder = "dermatologic_diseases/llm_text"
file_name = "*.txt"
table = "dermatologic_disease"

schema = [
  bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("physical_description", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("causes", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("treatments", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("_data_source", "STRING", mode="REQUIRED", default_value_expression="'dermatologic_disease'"),
  bigquery.SchemaField("_load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
]

create_load_table_from_json(folder, file_name, table, schema)

Created table dermatologic_disease
Loaded 370 rows.


#### Create and load `sephora_products`

In [ ]:
folder = "sephora_products"
file_name = "*.csv"
table = "sephora_product"
delimiter = ","

schema = [
  bigquery.SchemaField("product_id", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("product_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("brand_id", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("brand_name", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("loves_count", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("rating", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("reviews", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("size", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("variation_type", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("variation_value", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("variation_desc", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("ingredients", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("price_usd", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("value_price_usd", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("sale_price_usd", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("limited_edition", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("new", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("online_only", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("out_of_stock", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("sephora_exclusive", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("highlights", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("primary_category", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("secondary_category", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("tertiary_category", "STRING", mode="NULLABLE"),
  bigquery.SchemaField("child_count", "INTEGER", mode="NULLABLE"),
  bigquery.SchemaField("child_max_price", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("child_min_price", "FLOAT", mode="NULLABLE"),
  bigquery.SchemaField("_data_source", "STRING", mode="REQUIRED", default_value_expression="'sephora_product'"),
  bigquery.SchemaField("_load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
]

create_load_table_from_csv(folder, file_name, table, schema, delimiter)

Created table sephora_product
[SchemaField('product_id', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('product_name', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('brand_id', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('brand_name', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('loves_count', 'INTEGER', 'NULLABLE', None, None, (), None), SchemaField('rating', 'FLOAT', 'NULLABLE', None, None, (), None), SchemaField('reviews', 'INTEGER', 'NULLABLE', None, None, (), None), SchemaField('size', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('variation_type', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('variation_value', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('variation_desc', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('ingredients', 'STRING', 'NULLABLE', None, None, (), None), SchemaField('price_usd', 'FLOAT', 'NULLABLE', None, None, (), None), SchemaField('value_price_usd', 'FLOAT', 'NULL

#### Create and load `skin_disease_pictures`

In [4]:
folder = "skin_disease_pictures/raw"
file_name = "*.json"
table = "skin_disease_picture"

schema = [
  bigquery.SchemaField("skin_condition_name", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("skin_color", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("severity", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("image_link", "STRING", mode="REQUIRED"),
  bigquery.SchemaField("_data_source", "STRING", mode="REQUIRED", default_value_expression="'skin_disease_picture'"),
  bigquery.SchemaField("_load_time", "TIMESTAMP", mode="REQUIRED", default_value_expression="CURRENT_TIMESTAMP"),
]

create_load_table_from_json(folder, file_name, table, schema)

Created table skin_disease_picture
Loaded 3821 rows.
